# Feature Engineering — Wildfire Risk Forecaster

Compute vegetation indices (NDVI, EVI), merge weather data, and create spatial grid features
for wildfire risk prediction.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from scipy.ndimage import uniform_filter

## Feature Engineering Steps

1. Compute NDVI and EVI from multispectral bands
2. Extract weather features (temp, humidity, wind speed) per grid cell
3. Create spatial lag features (neighbor-averaged risk)
4. Generate topographic features (elevation, slope, aspect)
5. Build final feature matrix for modeling

In [ ]:
# Compute NDVI and EVI from satellite bands
with rasterio.open('../data/raw/multispectral.tif') as src:
    red = src.read(3).astype(float)
    nir = src.read(4).astype(float)
    blue = src.read(1).astype(float)

ndvi = (nir - red) / (nir + red + 1e-8)
evi = 2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1 + 1e-8)
print(f'NDVI shape: {ndvi.shape}, range: [{ndvi.min():.3f}, {ndvi.max():.3f}]')
print(f'EVI shape: {evi.shape}, range: [{evi.min():.3f}, {evi.max():.3f}]')

In [ ]:
# Merge weather data with spatial grid
weather = pd.read_csv('../data/raw/weather_stations.csv', parse_dates=['date'])
grid = gpd.read_file('../data/processed/spatial_grid.shp')

# Spatial join: assign weather to nearest grid cell
weather_gdf = gpd.GeoDataFrame(weather, geometry=gpd.points_from_xy(weather.lon, weather.lat))
merged = gpd.sjoin_nearest(grid, weather_gdf, how='left')

# Rolling statistics
for col in ['temperature', 'humidity', 'wind_speed']:
    merged[f'{col}_7d_mean'] = merged.groupby('grid_id')[col].transform(lambda x: x.rolling(7).mean())

print(f'Feature matrix shape: {merged.shape}')
merged.head()